In [90]:
import pandas as pd
import numpy as np

### Function to read the .txt files into a DataFrame

In [91]:

df_list = []
file_locs = ['../scraped_data/data.txt','../scraped_data/cardekho.txt']
for file in file_locs:
    with open(file, 'r', encoding='utf-8') as rf:
        for line in rf:
            inp = eval(line)
            main_dict = {}

            for key, value in inp.items():
                if(key == 'info'):
                    for keyinfo,valueinfo in inp[key].items():
                        main_dict[keyinfo] = valueinfo   
                else:
                        main_dict[key] = value
            df_list.append(main_dict)

df = pd.DataFrame(df_list)

# Basic Overview of the Data

In [92]:
df.head()

,model,price,price_score,plate,Reg. year,Fuel,KM driven,Transmission,Engine capacity,Ownership,Make year,Spare key,Reg number,Insurance,Insurance type
0,2020 Jeep Compass LONGITUDE PLUS 1.4 PETROL AT,₹8.50 lakh,STEAL DEAL according to our Price Indicator.,DL-10,Jul 2020,Petrol,"116,562 km",Automatic,1368cc,1st,Feb 2020,No,******8927,Nov 2026,3rd Party
1,2016 Mahindra XUV500 W6 AT 1.99,₹4.51 lakh,FAIR PRICE according to our Price Indicator.,DL-10,Aug 2016,Diesel,"238,933 km",Automatic,1997cc,1st,Jun 2016,No,******6357,NaN,NaN
2,2012 Hyundai Eon D-LITE,₹0.84 lakh,STEAL DEAL according to our Price Indicator.,UP-15,Jan 2013,CNG,"94,507 km",Manual,814cc,2nd,Jan 2012,No,******2716,Nov 2026,3rd Party
3,2013 Hyundai i20 SPORTZ 1.2,₹1.81 lakh,STEAL DEAL according to our Price Indicator.,DL-10,Mar 2013,CNG,"120,121 km",Manual,1197cc,2nd,Feb 2013,No,******7982,Dec 2026,3rd Party
4,2014 Honda City 1.5L I-VTEC SV,₹3.07 lakh,STEAL DEAL according to our Price Indicator.,DL-4C,May 2014,Petrol,"77,884 km",Manual,1497cc,2nd,Apr 2014,No,******9657,NaN,NaN


In [93]:
print(f"There are {df.duplicated().value_counts().iloc[1]} duplicated rows.")

There are 2351 duplicated rows.


# Modifying Datatypes :

In [94]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14928 entries, 0 to 14927
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   model            14928 non-null  object
 1   price            14928 non-null  object
 2   price_score      14628 non-null  object
 3   plate            14928 non-null  object
 4   Reg. year        14928 non-null  object
 5   Fuel             14928 non-null  object
 6   KM driven        14928 non-null  object
 7   Transmission     14928 non-null  object
 8   Engine capacity  14634 non-null  object
 9   Ownership        14928 non-null  object
 10  Make year        14928 non-null  object
 11  Spare key        9165 non-null   object
 12  Reg number       9165 non-null   object
 13  Insurance        4113 non-null   object
 14  Insurance type   9876 non-null   object
dtypes: object(15)
memory usage: 1.7+ MB


In [95]:
df.head()

,model,price,price_score,plate,Reg. year,Fuel,KM driven,Transmission,Engine capacity,Ownership,Make year,Spare key,Reg number,Insurance,Insurance type
0,2020 Jeep Compass LONGITUDE PLUS 1.4 PETROL AT,₹8.50 lakh,STEAL DEAL according to our Price Indicator.,DL-10,Jul 2020,Petrol,"116,562 km",Automatic,1368cc,1st,Feb 2020,No,******8927,Nov 2026,3rd Party
1,2016 Mahindra XUV500 W6 AT 1.99,₹4.51 lakh,FAIR PRICE according to our Price Indicator.,DL-10,Aug 2016,Diesel,"238,933 km",Automatic,1997cc,1st,Jun 2016,No,******6357,NaN,NaN
2,2012 Hyundai Eon D-LITE,₹0.84 lakh,STEAL DEAL according to our Price Indicator.,UP-15,Jan 2013,CNG,"94,507 km",Manual,814cc,2nd,Jan 2012,No,******2716,Nov 2026,3rd Party
3,2013 Hyundai i20 SPORTZ 1.2,₹1.81 lakh,STEAL DEAL according to our Price Indicator.,DL-10,Mar 2013,CNG,"120,121 km",Manual,1197cc,2nd,Feb 2013,No,******7982,Dec 2026,3rd Party
4,2014 Honda City 1.5L I-VTEC SV,₹3.07 lakh,STEAL DEAL according to our Price Indicator.,DL-4C,May 2014,Petrol,"77,884 km",Manual,1497cc,2nd,Apr 2014,No,******9657,NaN,NaN


- `model` to only include the main name of the model. 
- `price`, `KM driven` and `Engine capacity` to be converted into a numeric format
- `price_score` to be trimmed down.
- `plate` to be standardised.
- `Reg year` and `Make year` to be converted into datetime objects.


In [96]:
# creating a copy:
data = df.copy()

In [ ]:
# trimming model column :
data['model'] = data['model'].str.upper()
data['model'] = data['model'].str.replace('\n',' ') # cleaning up the newlines
data['model'] = data['model'].str.split(' ').map(lambda x: x[1]+' '+x[2] if x[2] != 'SUZUKI' else x[1]+' '+x[2]+' '+x[3])
data.loc[data['model'] == 'MARUTI SUZUKI SWIFT','model'] = 'MARUTI SWIFT' # maruti suzuki swift and maruti suzuki are the same car.
data['model'].value_counts()[:10] 

model
HONDA CITY       759
HYUNDAI CRETA    750
MARUTI SWIFT     688
KIA SELTOS       508
HYUNDAI GRAND    488
HONDA AMAZE      471
TATA NEXON       410
MARUTI WAGON     406
HYUNDAI I10      383
FORD ECOSPORT    333
Name: count, dtype: int64

In [98]:
# replacing price column :
def price_conversion(x):
    x = x.lower().replace('₹','')

    if(x.endswith('lakh')):
        x = x.replace('lakh','')
        x = float(x)*100000

    elif(x.endswith('thousand')):
        x = x.replace('thousand','')
        x = float(x)*1000
    
    return(x)

data['price'] = data['price'].map(price_conversion)

In [99]:
# converting KM driven and engine capacity column:
data['KM driven'] = data['KM driven'].str.replace(r'[A-Za-z,\s]+', '', regex=True).astype('int')
data['Engine capacity'] = data['Engine capacity'].str.strip().str.replace('cc','').astype('Float64')


In [100]:
# trimming price_score column :
data['price_score'] = data['price_score'].replace('NA', np.nan)
data.loc[data['price_score'].notna(), 'price_score'] = data.loc[data['price_score'].notna(), 'price_score'].str.split(' ').map(lambda x: x[0]+' '+x[1])

In [101]:
# converting Reg. year and Make year
data['Reg. year'] = pd.to_datetime(data['Reg. year']).dt.year
data['Make year'] = pd.to_datetime(data['Make year']).dt.year

C:\Users\sisfi\AppData\Local\Temp\ipykernel_17980\4109392050.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Reg. year'] = pd.to_datetime(data['Reg. year']).dt.year
C:\Users\sisfi\AppData\Local\Temp\ipykernel_17980\4109392050.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Make year'] = pd.to_datetime(data['Make year']).dt.year


In [115]:
# map plates :
data['plate'].value_counts()
data['plate'] = data['plate'].replace(r'[0-9-]+','', regex=True)
mappings = {'New Delhi': 'DL', 'Noida':'UP', 'DLC':'DL', 'DLS':'DL', 'DLD':'DL',
            'Gurgaon':'HR', 'Faridabad':'HR', 'Ghaziabad':'UP',
            'ballabhgarh': 'HR', 'Lucknow':'UP', 'palwal':'HR'
            }
abc = data['plate'].replace(mappings).value_counts()
abc[len(abc.index) > 2]

KeyError: True

# Examining Nulls :

In [103]:
df.isna().sum()

model                  0
price                  0
price_score          300
plate                  0
Reg. year              0
Fuel                   0
KM driven              0
Transmission           0
Engine capacity      294
Ownership              0
Make year              0
Spare key           5763
Reg number          5763
Insurance          10815
Insurance type      5052
dtype: int64

- `Spare Key` and `Reg Number` both have same number of null values, because they have been sourced from CarDekho, which does not provide this info.
- `price_score` field will have more nulls than depicted, because this info is not given in the CarDekho records either.
- `Engine capacity` nulls can be interpolated from other cars of the same model.
- `Insurance type` has less number of nulls than `Insurance`, because CarDekho data doesn't provide data for the 'Insurance' column.

In [104]:
# Replacing Insurance type nulls w/ unknown:
data.loc[data['Insurance type'].isna(),'Insurance type'] = 'Unknown'

In [105]:
# Replacing Engine capacity with other cars of the same make :
data['Engine capacity'] = data['Engine capacity'].fillna(
    data.groupby(['model', 'Make year'])['Engine capacity'].transform('mean'))

In [106]:
# for cars where model + year are too limiting :

data['Engine capacity'] = data['Engine capacity'].fillna(
    data.groupby('model')['Engine capacity'].transform('mean')
)

In [107]:
data[data['Engine capacity'].isna()]

,model,price,price_score,plate,Reg. year,Fuel,KM driven,Transmission,Engine capacity,Ownership,Make year,Spare key,Reg number,Insurance,Insurance type
963,MG ZS,1623000.0,GREAT PRICE,DL,2025,Electric,18403,Automatic,<NA>,1st,2024,No,******8399,Feb 2028,3rd Party
2034,MG ZS,1650000.0,HIGH PRICE,DLE,2024,Electric,49930,Automatic,<NA>,1st,2024,Yes,*****4683,Apr 2027,3rd Party
2793,CITROEN E,807000.0,FAIR PRICE,DLE,2023,Electric,48639,Automatic,<NA>,1st,2023,Yes,*****1395,NaN,Unknown
2898,MG COMET,622000.0,STEAL DEAL,DLE,2024,Electric,19780,Automatic,<NA>,1st,2024,Yes,*****5547,Jul 2027,3rd Party
2900,MG ZS,1435000.0,STEAL DEAL,DLE,2024,Electric,35917,Automatic,<NA>,1st,2024,No,*****1779,Jul 2027,3rd Party
3691,MG ZS,1200000.0,STEAL DEAL,DLG,2022,Electric,86273,Automatic,<NA>,1st,2022,No,*****0672,NaN,Unknown
6881,MG ZS,1200000.0,STEAL DEAL,DLG,2022,Electric,86273,Automatic,<NA>,1st,2022,No,*****0672,NaN,Unknown
8547,MAHINDRA XUV400,1620000.0,FAIR PRICE,DLE,2025,Electric,11869,Automatic,<NA>,1st,2025,Yes,DL6E*****,NaN,Unknown


There is a sparsity of data for the above models, meaning that no accurate model can be created for them.

# Dropping Unnecessary Rows and Columns :

In [108]:
data.drop_duplicates(inplace=True, keep='first')
data.shape[0]

12429

In [109]:
del data['Reg number']
del data['Spare key']
del data['Insurance']

In [110]:
data.head()

,model,price,price_score,plate,Reg. year,Fuel,KM driven,Transmission,Engine capacity,Ownership,Make year,Insurance type
0,JEEP COMPASS,850000.0,STEAL DEAL,DL,2020,Petrol,116562,Automatic,1368.0,1st,2020,3rd Party
1,MAHINDRA XUV500,451000.0,FAIR PRICE,DL,2016,Diesel,238933,Automatic,1997.0,1st,2016,Unknown
2,HYUNDAI EON,84000.0,STEAL DEAL,UP,2013,CNG,94507,Manual,814.0,2nd,2012,3rd Party
3,HYUNDAI I20,181000.0,STEAL DEAL,DL,2013,CNG,120121,Manual,1197.0,2nd,2013,3rd Party
4,HONDA CITY,307000.0,STEAL DEAL,DLC,2014,Petrol,77884,Manual,1497.0,2nd,2014,Unknown
